# Interactive plotting with `ehtim.plotting.interactive`

A Plotly-based companion to the matplotlib `comp_plots`/`summary_plots` defaults.
Same data, more interactivity: click legends to toggle baselines, drag to zoom,
hover for per-point hovertext, and use the toolbar to color baselines selectively
or fall back to plotly's default hide behavior.

This notebook walks through every public function — `plot_bl`, `plotall`,
`plot_gains`, `dashboard` — plus saving figures to standalone HTML and the
Color toolbar.

Requires `plotly` (in the `dev` extra, or `pip install plotly`).


## Setup

Load a sample uvfits + image; we'll use these throughout.

In [ ]:
import ehtim as eh
from ehtim.plotting import interactive

obs = eh.obsdata.load_uvfits('../data/sample.uvfits')
im = eh.image.load_txt('../models/avery_sgra_eofn.txt')
print(obs)


## `plot_bl` — one baseline at a time

Plot any visibility field for a single (site1, site2) pair against time.
Returns a Plotly Figure you can either display inline or save to HTML.

In [ ]:
interactive.plot_bl(obs, 'ALMA', 'LMT', 'amp')


Other useful fields: `'phase'`, `'snr'`, `'uvdist'`, `'sigma'`.
For a closed-RR/LL polarimetric obs, you can also pull individual hand
visibilities (`'rrvis'`, `'llvis'`, `'rlvis'`, `'lrvis'`).

In [ ]:
interactive.plot_bl(obs, 'ALMA', 'SMT', 'phase')


## `plotall` — any field against any other

The general two-field scatter. Pass `tag_bl=True` (the default) to break the
data into per-baseline traces colored from a palette, or `tag_bl=False` to
pool all points into a single trace.

In [ ]:
# Radplot: amplitude vs baseline length, per baseline.
interactive.plotall(obs, 'uvdist', 'amp')


### UV coverage

`plotall(obs, 'u', 'v')` is the UV-coverage view. The plot is locked to
a square aspect with a symmetric range driven by max(|u|, |v|), and the
axis values auto-scale to Gλ or Mλ based on the longest baseline. Pass
`conj=True` to overlay the Hermitian conjugate half.

In [ ]:
interactive.plotall(obs, 'u', 'v', conj=True)


### Closure quantities

Closure phase and log closure amplitude work the same way — just unpack
them via Obsdata's helpers and feed to `plot_bl` or use the dashboard
(see below).

## `plot_gains` — caltable gain time series

Drop in a calibration table from `self_cal` or `network_cal` to see per-site
amplitude or phase gains. `pol='R'` or `'L'` picks the channel; `gain_type`
selects amplitude vs phase. (TODO: schema-coupled — `R`/`L` become
`pol1`/`pol2` once the mixed-pol cal table lands.)

In [ ]:
from ehtim.calibrating.self_cal import self_cal

caltable = self_cal(
    obs, im, method='both', caltable=True,
    processes=2, show_solution=False,
)
interactive.plot_gains(caltable, gain_type='amp', pol='R')


## `dashboard` — image + data + gains + D-terms in one view

The main workhorse: a 2×2 dashboard summarizing a reconstruction.

- **Top left**: Stokes-I image (in μas RA/Dec offsets) with optional EVPA
  polarization ticks colored by fractional polarization. Toggle the
  polarization overlay with the "Toggle polarization" button.
- **Top right**: data product selector — dropdown switches between
  amp/vis/phase vs uvdist, chi² residual, closure phase vs time or
  triangle area, log closure amp vs time or quadrangle area.
- **Bottom left**: per-site gain time series.
- **Bottom right**: D-terms in the complex plane.

In [ ]:
# Add some fake linear polarization so the dashboard's pol overlay has something to show.
im_pol = im.copy()
I2d = im.imvec.reshape(im.ydim, im.xdim)
im_pol.add_qu(0.1 * I2d, 0.05 * I2d)

interactive.dashboard(im_pol, obs, caltable)


## Color toolbar

`plot_bl(tag_bl=True)`, `plotall(tag_bl=True)`, and the dashboard panels
add a three-button toolbar above the figure:

- **Color**: click to enter Color mode. While on, clicking a legend entry
  paints (or un-paints) that single trace instead of plotly's default hide.
- **Color all**: paint every baseline at once.
- **Show all / reset**: return everything to gray + visible.

Color mode is independent from plotly's built-in hide behavior — turn
Color mode off and legend clicks revert to the default toggle.

## Saving to HTML

`fig.write_html('plot.html')` works as you'd expect (we patch the
returned figure so the Color toolbar JS is embedded automatically) — or
use `interactive.write_html(fig, path)` for the same behavior with an
explicit path argument.

The exported PNG button on the modebar saves at 3× scale (≈ 300 dpi)
and sizes the canvas to fit the legend.

In [ ]:
fig = interactive.plotall(obs, 'u', 'v')
fig.write_html('/tmp/uv_coverage.html', include_plotlyjs='cdn')
print('saved /tmp/uv_coverage.html')


## Inline display in Jupyter

By default, the last expression in a cell renders the figure inline.
For the Color toolbar to appear inside the notebook output, use
`interactive.display(fig)` instead — it injects the toolbar JS the
same way `write_html` does.

In [ ]:
fig = interactive.plotall(obs, 'uvdist', 'amp')
interactive.display(fig)
